In [ ]:
import shutil, json, time
from pathlib import Path

TRAIN_DIR = Path("/kaggle/input/competitions/rsna-knee-abnormality-detection/train_series")
DATASET_DIR = Path("/kaggle/working/dicom_payload")
DATASET_DIR.mkdir(parents=True, exist_ok=True)

study_uids = sorted(p.name for p in TRAIN_DIR.iterdir() if p.is_dir())[:3]
print(f"Processing {len(study_uids)} studies...")

for uid in study_uids:
    src = TRAIN_DIR / uid
    dst = DATASET_DIR / uid
    shutil.copytree(src, dst)
    n = sum(1 for _ in dst.rglob("*.dcm"))
    print(f"  {uid}: {n} files")

(DATASET_DIR / "manifest.json").write_text(json.dumps({
    "created_at": time.strftime("%Y-%m-%d %H:%M:%S"),
    "n_studies": len(study_uids),
    "study_uids": study_uids,
}, indent=2))
print(f"\nDataset payload at {DATASET_DIR}")
print(f"Total size: {sum(f.stat().st_size for f in DATASET_DIR.rglob('*') if f.is_file()) / 1024 / 1024:.1f} MB")
print(f"File count: {sum(1 for f in DATASET_DIR.rglob('*') if f.is_file())}")

# Try the modern kagglesdk
print("\nTrying kagglesdk...")
try:
    from kagglesdk import KaggleClient
    client = KaggleClient()
    print("kagglesdk available")
    # We'd need to call dataset_create here, but kagglesdk doesn't expose this directly
except Exception as e:
    print(f"kagglesdk err: {e}")

# Try installing and using kaggle CLI from inside the kernel
import subprocess, os
print("\nLooking for kaggle credentials...")
for path in [Path.home() / '.kaggle' / 'kaggle.json', Path('/root/.kaggle/kaggle.json')]:
    if path.exists():
        print(f"Found: {path}")
        print(path.read_text()[:200])
    else:
        print(f"Not found: {path}")

# Try subprocess with the kernel's own token
print("\nTrying kaggle CLI via subprocess with kernel token...")
env = os.environ.copy()
# Check available auth mechanisms
print("KAGGLE_USER_SECRETS_TOKEN present:", 'KAGGLE_USER_SECRETS_TOKEN' in env)
print("KAGGLE_API_V1_TOKEN path:", env.get('KAGGLE_API_V1_TOKEN'))

# Try a tiny kaggle call
try:
    r = subprocess.run(['kaggle', 'kernels', 'list', '--mine', '--page-size', '3'],
                       capture_output=True, text=True, timeout=20, env=env)
    print(f"  rc: {r.returncode}")
    print(f"  stdout: {r.stdout[:300]}")
    print(f"  stderr: {r.stderr[:300]}")
except subprocess.TimeoutExpired:
    print("  TIMEOUT")
